# 8000 Ridge: отдельная модель на маршрут и горизонт

Автономный ноутбук для Kaggle и локального Jupyter. Входы: `train_solo_track.parquet` и `test_solo_track.parquet`. Train использует все получасовые старты. Val — 14 полных дней перед test, test — последние 14 полных дней; оба только со стартом 10:30. Все границы проверяются также по времени метки. Цель — прямой `target_1h`, прогноз ограничен снизу нулём.

Сначала Optuna подбирает alpha отдельно для каждой пары только на train/val. После фиксации всех 8000 настроек Ridge обучаются на train + val и оцениваются на test. Только после сохранения этой оценки начинается переобучение на всей размеченной истории. В финальном refit используются все получасовые старты с известной меткой, включая дополнительные старты дней val/test и последний неполный день.

Признаки перенесены из `feature_engineering.ipynb`: локальные лаги, статусы, окна, сезонные профили, сетевые суммы и два корреляционных лага. Календарь — только sin/cos дня недели и получаса суток. `profile28`, `overlap`, идентификаторы и временные метки в модель не входят. Повторяющиеся источники исключены; постоянные признаки исключаются внутри каждого fit. Начальная неполная история сохраняется, пропуски заполняются по обучающей части. Все исторические источники ограничены собственным cutoff строки.

Установите зависимости в активное окружение, если они отсутствуют: `%pip install "numpy>=1.26" "pandas>=2.2" "scipy>=1.12" "scikit-learn>=1.6" "optuna>=4" "pyarrow>=15" "joblib>=1.4" threadpoolctl psutil`. Затем перезапустите kernel. Для локального запуска удобно отдельное окружение Python 3.11/3.12. Для GPU-режима нужен CuPy под вашу CUDA: для CUDA 12 установите `%pip install cupy-cuda12x`, для CUDA 13 — `%pip install cupy-cuda13x`. Не устанавливайте несколько сборок CuPy одновременно. Перезапустите kernel после установки. На Kaggle включите NVIDIA GPU в настройках ноутбука.

Признаки готовятся двумя CPU-потоками с отдельным состоянием маршрутов. GPU обрабатывает пакет независимых пар маршрут × горизонт: матричные произведения, разложения, решения Ridge, прогнозы и метрики. Optuna работает через ask/tell: в каждом раунде одна попытка на каждую активную study. Разложение переиспользуется для разных alpha. Видеопамять и RAM ограничивают размер пакета; при CUDA OOM пакет делится пополам.

По умолчанию DEVICE='cuda', MAX_BATCH_SIZE=16, FEATURE_WORKERS=2. DEVICE='cpu' запускает тот же пакетный алгоритм через NumPy; DEVICE='auto' явно сообщает о переходе на CPU при недоступном CUDA. Все матричные вычисления — float64 ради устойчивости при коррелированных признаках и малых alpha. Это может ограничивать ускорение на потребительских GPU; быстродействие необходимо измерять.

Финальные файлы содержат обычную sklearn Ridge с коэффициентами, вычисленными на выбранном устройстве; CuPy для загрузки и инференса этих файлов не нужен. Первые две пары каждого этапа автоматически сверяются с CPU sklearn Ridge. Для всех пар проверяется невязка решения, а при подборе — согласованность спектрального решения с прямым решением Ridge. CUDA-исполнение необходимо проверить на целевом устройстве: локальная проверка разработчика выполнена через NumPy на синтетических данных без NVIDIA GPU.


In [1]:
from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import platform
import time
import zipfile
import threading
import types
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
import joblib
import numpy as np
import pandas as pd
import optuna
import psutil
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits
from IPython.display import display

TRAIN_PATH = Path('data/train_solo_track.parquet')
TEST_PATH = Path('data/test_solo_track.parquet')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MODEL_ROOT = OUTPUT_ROOT / 'models' / 'ridges'
RUN_ROOT = OUTPUT_ROOT / 'artifacts' / 'ridge_by_route_horizon'
N_TRIALS = 30
ALPHA_LOW = 1e-4
ALPHA_HIGH = 1e6
SEED = 42
VAL_DAYS = 14
TEST_DAYS = 14
EXPECTED_ROUTES = 1000
BLAS_THREADS = 1
DEVICE = 'cuda'
MAX_BATCH_SIZE = 16
FEATURE_WORKERS = 2
HOST_MEMORY_FRACTION = 0.25
GPU_MEMORY_FRACTION = 0.60
CPU_REFERENCE_PAIRS = 2
CHECKPOINT_EVERY = 5
PRINT_EVERY = 25
CREATE_ARCHIVE = True
PIPELINE_VERSION = 'route_ridges_batched_v2'
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']
optuna.logging.set_verbosity(optuna.logging.WARNING)
if N_TRIALS < 1 or not 0 < ALPHA_LOW < ALPHA_HIGH or CHECKPOINT_EVERY < 1:
    raise ValueError('Некорректные настройки Optuna')
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    found_train = list(Path('/kaggle/input').rglob(TRAIN_PATH.name))
    found_test = list(Path('/kaggle/input').rglob(TEST_PATH.name))
    if len(found_train) != 1 or len(found_test) != 1:
        raise ValueError('Укажите TRAIN_PATH и TEST_PATH вручную: найдены неоднозначные входные файлы')
    TRAIN_PATH, TEST_PATH = found_train[0], found_test[0]
if not TRAIN_PATH.is_file() or not TEST_PATH.is_file():
    raise FileNotFoundError('Укажите пути к двум исходным parquet')
print('CPU:', platform.processor() or platform.machine())
print('RAM GiB:', round(psutil.virtual_memory().total / 2**30, 1), 'Доступно:', round(psutil.virtual_memory().available / 2**30, 1))
if DEVICE not in ['cuda', 'cpu', 'auto']:
    raise ValueError("DEVICE должен быть 'cuda', 'cpu' или 'auto'")
if MAX_BATCH_SIZE < 1 or FEATURE_WORKERS < 1 or BLAS_THREADS < 1 or CPU_REFERENCE_PAIRS < 1:
    raise ValueError('Размер пакета, число потоков и число контрольных пар должны быть положительными')
if not 0 < HOST_MEMORY_FRACTION <= 0.5 or not 0 < GPU_MEMORY_FRACTION < 1:
    raise ValueError('Некорректные доли памяти')
cp = None
cupyx = None
BACKEND = 'cpu'
if DEVICE in ['cuda', 'auto']:
    try:
        import cupy as cp
        import cupyx
        if cp.cuda.runtime.getDeviceCount() < 1:
            raise RuntimeError('CUDA-устройства не найдены')
        with cupyx.errstate(linalg='raise'):
            probe = cp.asarray([[[2.0, 0.0], [0.0, 3.0]]], dtype=cp.float64)
            probe_values, probe_vectors = cp.linalg.eigh(probe)
            probe_solution = cp.linalg.solve(probe, cp.ones((1, 2, 1), dtype=cp.float64))
            if not np.allclose(cp.asnumpy(probe_values), [[2.0, 3.0]]) or not np.allclose(cp.asnumpy(probe_solution).ravel(), [0.5, 1 / 3]):
                raise RuntimeError('CUDA-проверка пакетной линейной алгебры не прошла')
        cp.cuda.get_current_stream().synchronize()
        del probe, probe_values, probe_vectors, probe_solution
        cp.get_default_memory_pool().free_all_blocks()
        BACKEND = 'cuda'
    except Exception as error:
        if DEVICE == 'cuda':
            raise RuntimeError("CUDA недоступна. Включите GPU и установите подходящий CuPy либо задайте DEVICE='cpu'. Причина: " + str(error)) from error
        print('DEVICE=auto: переход на CPU. Причина:', str(error))
        cp = None
        cupyx = None
xp = cp if BACKEND == 'cuda' else np
if BACKEND == 'cuda':
    properties = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
    gpu_name = properties['name']
    if isinstance(gpu_name, bytes):
        gpu_name = gpu_name.decode()
    free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
    print('GPU:', gpu_name, 'VRAM GiB:', round(total_bytes / 2**30, 2), 'Свободно:', round(free_bytes / 2**30, 2))
print('Backend:', BACKEND, 'Максимум моделей в пакете:', MAX_BATCH_SIZE, 'Потоков подготовки:', FEATURE_WORKERS)
print('BLAS threads:', BLAS_THREADS, 'Матричная точность: float64')


c:\Users\artem\projects\experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
RAM GiB: 15.7 Доступно: 5.6


c:\Users\artem\projects\experiments\.venv\Lib\site-packages\cupy\_environment.py:286: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


RuntimeError: CUDA недоступна. Включите GPU и установите подходящий CuPy либо задайте DEVICE='cpu'. Причина: Failure finding "known nvJitLink DLL": No such file: known nvJitLink DLL, No such file: known nvJitLink DLL


## Входные данные, границы и продолжение

Подбор и отчёты сохраняются в `artifacts/ridge_by_route_horizon`, финальные 8000 моделей — в `models/ridges/h1…h8/route_XXXX.joblib`. Индекс маршрута сопоставлен с исходным route_id в `route_index.csv`. Один процесс обрабатывает этот каталог; одновременные запуски в один каталог не поддерживаются.

Повторный запуск с теми же входами и настройками пропускает завершённые этапы и продолжает Optuna с последнего checkpoint (раз в пять trials и по завершении пары). При изменении входов, настроек или версий библиотек используйте новые RUN_ROOT и MODEL_ROOT. Для продолжения в новой Kaggle-сессии сначала восстановите обе папки в доступное для записи место. Checkpoint не переживает удаление самой сессии без сохранения её файлов.
Версия пакетного алгоритма имеет новый manifest и не продолжает старые CPU-checkpoints v1. Если старые результаты уже существуют, сохраните их отдельно и задайте новые RUN_ROOT и MODEL_ROOT. Смена CPU/CUDA также требует отдельного запуска. MAX_BATCH_SIZE, FEATURE_WORKERS и доли памяти можно уменьшать при продолжении текущего запуска.


In [ ]:
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def atomic_joblib(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    joblib.dump(value, temporary, compress=3)
    temporary.replace(path)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def scalar(value):
    return value.item() if isinstance(value, np.generic) else value

history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES])
future_requests = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
history['timestamp'] = pd.to_datetime(history.timestamp)
future_requests['timestamp'] = pd.to_datetime(future_requests.timestamp)
history = history.sort_values(['timestamp', 'route_id']).reset_index(drop=True)
times = pd.DatetimeIndex(history.timestamp.unique())
all_routes = np.sort(history.route_id.unique())
if history.isna().any().any() or future_requests.isna().any().any():
    raise ValueError('В исходных данных есть пропуски')
if len(all_routes) != EXPECTED_ROUTES:
    raise ValueError(f'Ожидалось {EXPECTED_ROUTES} маршрутов, найдено {len(all_routes)}')
if history.duplicated(['route_id', 'timestamp']).any() or len(history) != len(times) * len(all_routes):
    raise ValueError('История не является полной панелью маршрутов')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
end = times[-1]
if (end.hour, end.minute) != (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([all_routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
actual = pd.MultiIndex.from_frame(future_requests[['route_id', 'timestamp']])
if not future_requests.id.is_unique or actual.has_duplicates or len(actual) != len(expected) or len(expected.difference(actual)):
    raise ValueError('Конкурсный test должен содержать все 8000 пар маршрут × горизонт без повторов')
all_values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(all_routes)) for name in [TARGET, *STATUSES]}
del history
gc.collect()
if not all(np.isfinite(array).all() for array in all_values.values()) or (all_values[TARGET] < 0).any():
    raise ValueError('Некорректные числовые значения')
full_dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(full_dates) <= VAL_DAYS + TEST_DAYS + 7:
    raise ValueError('Недостаточно истории для разбиения')
test_start = full_dates[-TEST_DAYS].normalize()
val_start = full_dates[-TEST_DAYS - VAL_DAYS].normalize()
split_table = pd.DataFrame([
    dict(period='train', first=str(times[0]), last_exclusive=str(val_start), origins='все получасы'),
    dict(period='val', first=str(val_start), last_exclusive=str(test_start), origins='10:30'),
    dict(period='test', first=str(test_start), last_exclusive=str(full_dates[-1].normalize() + pd.Timedelta(days=1)), origins='10:30'),
])
display(split_table)
network_signals = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in all_values.items()}
versions = {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'optuna', 'pyarrow', 'joblib']}
if BACKEND == 'cuda':
    versions['cupy'] = cp.__version__
    versions['cuda_runtime'] = str(cp.cuda.runtime.runtimeGetVersion())
config = dict(backend=BACKEND, matrix_dtype='float64', pipeline_version=PIPELINE_VERSION, n_trials=N_TRIALS, alpha_low=ALPHA_LOW, alpha_high=ALPHA_HIGH,
              seed=SEED, val_days=VAL_DAYS, test_days=TEST_DAYS, expected_routes=EXPECTED_ROUTES,
              val_start=str(val_start), test_start=str(test_start), end=str(end),
              lag_max_hours=LAG_MAX_LOOKBACK_HOURS, lag_min_pairs=LAG_MIN_PAIRS, lag_min_correlation=LAG_MIN_CORRELATION,
              window_offsets=list(WINDOW_OFFSETS), versions=versions,
              train_sha256=file_hash(TRAIN_PATH), test_sha256=file_hash(TEST_PATH))
run_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
for folder in [RUN_ROOT, MODEL_ROOT]:
    manifest_path = folder / 'manifest.json'
    if manifest_path.exists():
        if json.loads(manifest_path.read_text())['run_id'] != run_id:
            raise ValueError(f'Настройки или данные изменились. Используйте новую папку: {folder}')
    elif folder.exists() and any(folder.iterdir()):
        raise ValueError(f'Папка не пуста и не имеет manifest: {folder}')
for folder in [RUN_ROOT, MODEL_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
    atomic_json(folder / 'manifest.json', dict(run_id=run_id, config=config))
for name in ['studies', 'params', 'trials', 'evaluation', 'full', 'lags', 'reports']:
    (RUN_ROOT / name).mkdir(exist_ok=True)
pd.DataFrame({'route_index': np.arange(len(all_routes)), 'route_id': all_routes}).to_csv(RUN_ROOT / 'reports' / 'route_index.csv', index=False)
split_table.to_csv(RUN_ROOT / 'reports' / 'splits.csv', index=False)
metric_display = display(pd.DataFrame(), display_id=True)
print('Всего пар:', len(all_routes) * 8, 'Trials:', len(all_routes) * 8 * N_TRIALS)


## Признаки одного маршрута

Статусы и цель используются только до cutoff включительно. Соседние сезонные слоты ±2 часа берутся в прошлые дни. Для последних значений доступны только сдвиги назад. Сетевые суммы рассчитаны по всем маршрутам того же доступного момента. Корреляционные лаги выбираются отдельно на train, train + val и всей истории; val/test не влияют на обучаемые преобразования своего этапа.

Названия warehouse/hub сохранены из исходного ноутбука и обозначают формулы. Суммы статусов не трактуются как число уникальных товаров. На val/test факты предыдущих дней становятся доступной историей следующего дня: это ежедневные прогнозы, а не единый прогноз на две недели.

In [ ]:
def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common


In [ ]:
def prepare_route(route_index):
    global values, routes, target, means, positions, extra_signals, hub_signal, residual_target, residual_signal
    routes = all_routes[route_index:route_index + 1]
    values = {name: array[:, route_index:route_index + 1] for name, array in all_values.items()}
    target = values[TARGET]
    positions = np.arange(len(times))
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum(extra_signals[name + '_hourly'] for name in STATUSES[:3])
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-5))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
    recent = means[TARGET, 4]
    usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
    extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    hub_signal = extra_signals['status_6_diff_hourly']
    for name, signal in network_signals.items():
        extra_signals[f'network_{name}_latest'] = signal
        extra_signals[f'network_{name}_change_1h'] = signal - past(signal, positions - 2)
    residual_target = remove_seasonality(target)
    residual_signal = remove_seasonality(hub_signal)


In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)

def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = {}
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    slot_number = dates.hour * 2 + dates.minute / 30
    calendar['half_hour_sin'] = np.sin(2 * np.pi * slot_number / 48)
    calendar['half_hour_cos'] = np.cos(2 * np.pi * slot_number / 48)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()


In [ ]:
def feature_columns(frame):
    return [name for name in frame.columns if name not in ['route_id', 'cutoff', 'timestamp', 'y', 'id']]

def route_frame(h, lag_table, phase):
    if phase == 'search':
        origins = np.flatnonzero(times + h * STEP < test_start)
    elif phase == 'evaluation':
        origins = np.flatnonzero((np.arange(len(times)) + h < len(times)) & (times.normalize() <= full_dates[-1].normalize()))
    elif phase == 'full':
        origins = np.arange(len(times) - h)
    else:
        raise ValueError('Неизвестный этап')
    frame = horizon_dataset(h, origins)
    frame['y'] = target[origins + h, 0]
    frame = add_correlation_features(frame, h, lag_table)
    frame[feature_columns(frame)] = frame[feature_columns(frame)].replace([np.inf, -np.inf], np.nan)
    return frame

def split_masks(frame):
    at_1030 = frame.cutoff.dt.hour.eq(10) & frame.cutoff.dt.minute.eq(30)
    return dict(
        train=frame.timestamp.lt(val_start),
        val=at_1030 & frame.cutoff.ge(val_start) & frame.timestamp.lt(test_start),
        train_val=frame.timestamp.lt(test_start),
        test=at_1030 & frame.cutoff.ge(test_start) & frame.cutoff.dt.normalize().le(full_dates[-1].normalize()),
    )

def lag_table_for(route_index, phase):
    boundary = {'search': val_start, 'evaluation': test_start, 'full': end + STEP}[phase]
    table = fit_lags(boundary)
    path = RUN_ROOT / 'lags' / f'{phase}_route_{route_index:04d}.csv'
    table.to_csv(path, index=False)
    return table

def prepare_matrix(train, columns):
    raw = train[columns].to_numpy(dtype=np.float64)
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    filled = imputer.fit_transform(raw)
    active = np.flatnonzero(np.ptp(filled, axis=0) > 0)
    if len(active) == 0:
        active = np.array([0], dtype=np.int64)
    scaler = StandardScaler()
    matrix = scaler.fit_transform(filled[:, active])
    preprocessing = dict(columns=columns, imputer=imputer, active_indices=active, scaler=scaler)
    return preprocessing, matrix

def transform_matrix(frame, preprocessing):
    filled = preprocessing['imputer'].transform(frame[preprocessing['columns']].to_numpy(dtype=np.float64))
    return preprocessing['scaler'].transform(filled[:, preprocessing['active_indices']])

def metric_values(y, prediction):
    y = np.asarray(y, dtype=np.float64)
    prediction = np.maximum(np.asarray(prediction, dtype=np.float64), 0)
    if len(y) == 0 or not np.isfinite(prediction).all() or not np.isfinite(y).all():
        raise ValueError('Пустой набор или нечисловой прогноз')
    error = prediction - y
    volume = float(y.sum())
    absolute = float(np.abs(error).sum())
    signed = float(error.sum())
    return dict(n=len(y), actual_sum=volume, prediction_sum=float(prediction.sum()), absolute_error=absolute,
                signed_error=signed, WAPE=absolute / volume if volume > 0 else None,
                abs_relative_bias=abs(signed) / volume if volume > 0 else None,
                score=(absolute + abs(signed)) / volume if volume > 0 else None, MAE=absolute / len(y))

def metric_row(y, prediction, period, **extra):
    return dict(period=period, **extra, **metric_values(y, prediction))

def fit_model(frame, alpha, columns):
    preprocessing, matrix = prepare_matrix(frame, columns)
    model = Ridge(alpha=float(alpha), fit_intercept=True, solver='cholesky')
    model.fit(matrix, frame.y.to_numpy(dtype=np.float64))
    return dict(preprocessing=preprocessing, model=model)

def predict_model(artifact, frame):
    return np.maximum(artifact['model'].predict(transform_matrix(frame, artifact['preprocessing'])), 0)

def pair_key(route_index, h):
    return f'route_{route_index:04d}_h{h}'


def to_host(value):
    return cp.asnumpy(value) if BACKEND == 'cuda' else np.asarray(value)

def synchronize():
    if BACKEND == 'cuda':
        cp.cuda.get_current_stream().synchronize()

def linalg_guard():
    return cupyx.errstate(linalg='raise') if BACKEND == 'cuda' else nullcontext()

FEATURE_FUNCTIONS = ['past', 'average_available', 'rolling', 'offset_name', 'add_window', 'common_features',
                     'prepare_route', 'calculate_dynamic_shoulder', 'correlation_block', 'remove_seasonality',
                     'fit_lags', 'add_correlation_features', 'horizon_dataset', 'feature_columns', 'route_frame', 'split_masks']
FEATURE_GLOBALS = ['np', 'pd', 'TARGET', 'STATUSES', 'STEP', 'WINDOW_OFFSETS', 'LAG_MAX_LOOKBACK_HOURS',
                   'LAG_MIN_PAIRS', 'LAG_MIN_CORRELATION', 'CORRELATION_COLUMNS', 'all_values', 'all_routes',
                   'network_signals', 'times', 'end', 'full_dates', 'val_start', 'test_start']
feature_local = threading.local()

class RouteContext:
    def __init__(self, route_index):
        self.route_index = route_index
        self.namespace = {name: globals()[name] for name in FEATURE_GLOBALS}
        self.namespace['__builtins__'] = __builtins__
        for name in FEATURE_FUNCTIONS:
            original = globals()[name]
            cloned = types.FunctionType(original.__code__, self.namespace, name, original.__defaults__, original.__closure__)
            cloned.__kwdefaults__ = original.__kwdefaults__
            self.namespace[name] = cloned
        self.namespace['prepare_route'](route_index)
        self.lags = {}

    def lag_table(self, phase):
        if phase not in self.lags:
            boundary = {'search': val_start, 'evaluation': test_start, 'full': end + STEP}[phase]
            self.lags[phase] = self.namespace['fit_lags'](boundary)
        return self.lags[phase]

def prepare_pair(task, phase):
    route_index, h = task
    context = getattr(feature_local, 'context', None)
    if context is None or context.route_index != route_index:
        context = RouteContext(route_index)
        feature_local.context = context
    table = context.lag_table(phase)
    frame = context.namespace['route_frame'](h, table, phase)
    masks = split_masks(frame)
    if phase == 'search':
        training = frame.loc[masks['train']]
        query = frame.loc[masks['val']]
        if len(query) != VAL_DAYS or training.empty or training.timestamp.max() >= val_start:
            raise ValueError('Нарушено разбиение train/val')
    elif phase == 'evaluation':
        training = frame.loc[masks['train_val']]
        query = frame.loc[masks['test']]
        if len(query) != TEST_DAYS or training.empty or training.timestamp.max() >= test_start:
            raise ValueError('Нарушено разбиение train + val/test')
    else:
        training = frame
        query = context.namespace['horizon_dataset'](h, np.array([len(times) - 1]))
        query = context.namespace['add_correlation_features'](query, h, table)
        query[feature_columns(query)] = query[feature_columns(query)].replace([np.inf, -np.inf], np.nan)
    preprocessing, matrix = prepare_matrix(training, feature_columns(frame))
    query_matrix = transform_matrix(query, preprocessing)
    x_mean = matrix.mean(axis=0)
    matrix -= x_mean
    query_matrix -= x_mean
    y = training.y.to_numpy(dtype=np.float64)
    columns = ['route_id', 'cutoff', 'timestamp'] + (['y'] if phase != 'full' else [])
    return dict(key=pair_key(route_index, h), route_index=route_index, route_id=scalar(all_routes[route_index]),
                horizon=h, phase=phase, preprocessing=preprocessing, x=matrix, x_mean=x_mean,
                y=y, y_mean=float(y.mean()), query_x=query_matrix,
                query_y=query.y.to_numpy(dtype=np.float64) if phase != 'full' else np.zeros(len(query)),
                query_frame=query[columns].copy(), original_train=training.timestamp.lt(val_start).to_numpy(),
                lag_table=table.copy(), fit_through=str(training.timestamp.max()))

def estimate_batch_size():
    columns = 512
    per_pair_host = len(times) * columns * 8 * 8
    host_budget = psutil.virtual_memory().available * HOST_MEMORY_FRACTION
    host_limit = max(1, int(host_budget // per_pair_host))
    gpu_limit = MAX_BATCH_SIZE
    if BACKEND == 'cuda':
        cp.get_default_memory_pool().free_all_blocks()
        free_bytes, _ = cp.cuda.runtime.memGetInfo()
        per_pair_gpu = 8 * (len(times) * columns * 2 + columns * columns * 12)
        gpu_limit = max(1, int(free_bytes * GPU_MEMORY_FRACTION // per_pair_gpu))
    return max(1, min(MAX_BATCH_SIZE, host_limit, gpu_limit))

def totals_to_metrics(values):
    count, volume, predicted, absolute, signed = [float(value) for value in values]
    if count < 1 or not np.isfinite(values).all():
        raise ValueError('Нечисловые или пустые пакетные метрики')
    return dict(n=int(round(count)), actual_sum=volume, prediction_sum=predicted, absolute_error=absolute,
                signed_error=signed, WAPE=absolute / volume if volume > 0 else None,
                abs_relative_bias=abs(signed) / volume if volume > 0 else None,
                score=(absolute + abs(signed)) / volume if volume > 0 else None, MAE=absolute / count)

class DeviceBatch:
    def __init__(self, items, spectral):
        self.items = items
        self.size = len(items)
        self.width = max(item['x'].shape[1] for item in items)
        rows = max(len(item['y']) for item in items)
        query_rows = max(len(item['query_y']) for item in items)
        self.x = xp.zeros((self.size, rows, self.width), dtype=xp.float64)
        self.q = xp.zeros((self.size, query_rows, self.width), dtype=xp.float64)
        self.y = xp.zeros((self.size, rows), dtype=xp.float64)
        self.qy = xp.zeros((self.size, query_rows), dtype=xp.float64)
        self.mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.original_mask = xp.zeros((self.size, rows), dtype=xp.bool_)
        self.query_mask = xp.zeros((self.size, query_rows), dtype=xp.bool_)
        self.y_mean = xp.asarray([item['y_mean'] for item in items], dtype=xp.float64)
        for index, item in enumerate(items):
            n, p = item['x'].shape
            qn = len(item['query_y'])
            self.x[index, :n, :p] = xp.asarray(item['x'])
            self.q[index, :qn, :p] = xp.asarray(item['query_x'])
            self.y[index, :n] = xp.asarray(item['y'])
            self.qy[index, :qn] = xp.asarray(item['query_y'])
            self.mask[index, :n] = True
            self.original_mask[index, :n] = xp.asarray(item['original_train'])
            self.query_mask[index, :qn] = True
        transposed = self.x.swapaxes(-1, -2)
        self.gram = transposed @ self.x
        centered_y = xp.where(self.mask, self.y - self.y_mean[:, None], 0.0)
        self.rhs = (transposed @ centered_y[:, :, None])[:, :, 0]
        if spectral:
            with linalg_guard():
                values, self.vectors = xp.linalg.eigh(self.gram)
            self.eigenvalues = xp.maximum(values, 0.0)
            self.projected = (self.vectors.swapaxes(-1, -2) @ self.rhs[:, :, None])[:, :, 0]
        synchronize()

    def coefficients(self, alphas, spectral):
        alpha = xp.asarray(alphas, dtype=xp.float64)
        if spectral:
            quotient = self.projected / (self.eigenvalues + alpha[:, None])
            result = (self.vectors @ quotient[:, :, None])[:, :, 0]
        else:
            system = self.gram.copy()
            index = xp.arange(self.width)
            system[:, index, index] += alpha[:, None]
            with linalg_guard():
                result = xp.linalg.solve(system, self.rhs[:, :, None])[:, :, 0]
        residual = (self.gram @ result[:, :, None])[:, :, 0] + alpha[:, None] * result - self.rhs
        relative = xp.sqrt(xp.sum(residual * residual, axis=1)) / xp.maximum(xp.sqrt(xp.sum(self.rhs * self.rhs, axis=1)), 1.0)
        if not bool(to_host(xp.isfinite(result).all())) or np.any(to_host(relative) > 1e-7):
            raise ValueError('Пакетное решение Ridge не прошло проверку невязки')
        return result

    def predictions(self, coefficients):
        train = xp.maximum((self.x @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        query = xp.maximum((self.q @ coefficients[:, :, None])[:, :, 0] + self.y_mean[:, None], 0.0)
        return train, query

    def reduce(self, prediction, actual, mask):
        error = xp.where(mask, prediction - actual, 0.0)
        totals = xp.stack([mask.sum(axis=1), xp.where(mask, actual, 0.0).sum(axis=1),
                           xp.where(mask, prediction, 0.0).sum(axis=1), xp.abs(error).sum(axis=1), error.sum(axis=1)], axis=1)
        return [totals_to_metrics(values) for values in to_host(totals)]

    def metrics(self, coefficients, include_original=False):
        train, query = self.predictions(coefficients)
        result = dict(train=self.reduce(train, self.y, self.mask))
        if self.items[0]['phase'] != 'full':
            result['query'] = self.reduce(query, self.qy, self.query_mask)
        if include_original:
            result['original_train'] = self.reduce(train, self.y, self.original_mask)
        return result, to_host(query)

def portable_model(item, coefficients):
    width = item['x'].shape[1]
    model = Ridge(alpha=float(item['alpha']), fit_intercept=True, solver='cholesky')
    model.coef_ = np.asarray(coefficients[:width], dtype=np.float64).copy()
    model.intercept_ = float(item['y_mean'] - item['x_mean'] @ model.coef_)
    model.n_features_in_ = width
    model.n_iter_ = None
    return dict(preprocessing=item['preprocessing'], model=model)

reference_counts = {'search': 0, 'evaluation': 0, 'full': 0}

def check_cpu_reference(item, coefficients, query_prediction):
    phase = item['phase']
    if reference_counts[phase] >= CPU_REFERENCE_PAIRS:
        return
    model = Ridge(alpha=item['alpha'], fit_intercept=True, solver='cholesky')
    model.fit(item['x'], item['y'])
    expected = np.maximum(model.predict(item['query_x']), 0.0)
    actual = query_prediction[:len(expected)]
    scale = max(1.0, float(np.abs(item['y']).max()))
    if not np.allclose(actual, expected, rtol=1e-6, atol=scale * 1e-7):
        raise ValueError(f"CPU sklearn Ridge и {BACKEND} расходятся: {item['key']}")
    if not np.allclose(coefficients[:len(model.coef_)], model.coef_, rtol=1e-5, atol=scale * 1e-7):
        raise ValueError('Коэффициенты пакетного Ridge отличаются от CPU-эталона')
    atomic_json(RUN_ROOT / 'reports' / f"numerical_check_{phase}_{item['key']}.json",
                dict(backend=BACKEND, phase=phase, key=item['key'], max_prediction_difference=float(np.max(np.abs(actual - expected)))))
    reference_counts[phase] += 1

def phase_pending(phase, route_index, h):
    key = pair_key(route_index, h)
    if phase == 'search':
        return not (RUN_ROOT / 'params' / f'{key}.json').exists()
    if phase == 'evaluation':
        return not ((RUN_ROOT / 'evaluation' / f'{key}.json').exists() and (RUN_ROOT / 'evaluation' / f'{key}.parquet').exists())
    return not ((RUN_ROOT / 'full' / f'{key}.json').exists() and (MODEL_ROOT / f'h{h}' / f'route_{route_index:04d}.joblib').exists())

def execute_with_retry(items, phase):
    retry = False
    try:
        process_device_batch(items, phase)
    except Exception as error:
        if BACKEND != 'cuda' or not isinstance(error, cp.cuda.memory.OutOfMemoryError):
            raise
        if len(items) == 1:
            raise RuntimeError('Одной модели не хватает VRAM. Освободите GPU-память или запустите новый CPU-запуск.') from error
        retry = True
    if retry:
        gc.collect()
        cp.get_default_memory_pool().free_all_blocks()
        middle = len(items) // 2
        print(f'CUDA OOM: пакет {len(items)} разделён на {middle} и {len(items) - middle}', flush=True)
        execute_with_retry(items[:middle], phase)
        execute_with_retry(items[middle:], phase)

def run_phase(phase):
    pending = [(route_index, h) for route_index in range(len(all_routes)) for h in range(1, 9) if phase_pending(phase, route_index, h)]
    total = len(all_routes) * 8
    skipped = total - len(pending)
    completed = 0
    started = time.perf_counter()
    timings = []
    print(f'{phase}: готовых {skipped}/{total}, backend={BACKEND}', flush=True)
    with ThreadPoolExecutor(max_workers=FEATURE_WORKERS) as executor:
        while completed < len(pending):
            batch_size = estimate_batch_size()
            tasks = pending[completed:completed + batch_size]
            begin_features = time.perf_counter()
            items = list(executor.map(lambda task: prepare_pair(task, phase), tasks))
            feature_seconds = time.perf_counter() - begin_features
            begin_compute = time.perf_counter()
            execute_with_retry(items, phase)
            synchronize()
            compute_seconds = time.perf_counter() - begin_compute
            for item in items:
                item['lag_table'].to_csv(RUN_ROOT / 'lags' / f"{phase}_route_{item['route_index']:04d}.csv", index=False)
            completed += len(items)
            elapsed = time.perf_counter() - started
            seconds_per_pair = elapsed / completed
            remaining = (len(pending) - completed) * seconds_per_pair
            row = dict(phase=phase, backend=BACKEND, batch_size=len(items), completed=completed, skipped=skipped,
                       feature_seconds=feature_seconds, compute_and_save_seconds=compute_seconds,
                       seconds_per_pair=seconds_per_pair, remaining_hours=remaining / 3600)
            timings.append(row)
            pd.DataFrame(timings).to_csv(RUN_ROOT / 'reports' / f'timings_{phase}.csv', index=False)
            print(f'{phase}: {completed + skipped}/{total}, пакет {len(items)}, {seconds_per_pair:.2f} с/пару, осталось около {remaining / 3600:.2f} ч', flush=True)
            del items
            gc.collect()


## Пакетный Optuna: только train и val

30 trials на пару, alpha от 1e-4 до 1e6 по логарифмической шкале. Цель — WAPE + |Relative Bias| по 14 точкам val этой пары; при нулевом объёме val используется MAE с явной отметкой. Test до фиксации всех настроек не оценивается.

Каждая study имеет свой seed и sampler. В одном раунде ask получает по одному alpha для каждой активной пары; GPU одновременно вычисляет коэффициенты всех моделей, прогнозы и метрики. Затем tell возвращает результат в соответствующую study. Последующие предложения одной пары используют только её уже законченные попытки. Разный размер пакета не меняет порядок предложений внутри каждой study.

Матрицы дополняются нулями до общей формы, но дополнительные строки исключены из метрик. Коэффициенты неактивных признаков не сохраняются. Медианы, масштабирование и выбор признаков остаются отдельными для каждой модели. Checkpoints сохраняются после tell каждые пять завершённых попыток и при завершении study. В файлах trials находятся метрики каждой попытки; обновляемая таблица отображает выбранные конфигурации последнего пакета.


In [ ]:
def load_study(item):
    path = RUN_ROOT / 'studies' / f"{item['key']}.joblib"
    if path.exists():
        study = joblib.load(path)
        if study.user_attrs.get('run_id') != run_id:
            raise ValueError('Optuna checkpoint относится к другому запуску')
        for trial in study.get_trials(deepcopy=False):
            if trial.state == optuna.trial.TrialState.RUNNING:
                study.tell(trial.number, state=optuna.trial.TrialState.FAIL)
    else:
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED + item['route_index'] * 8 + item['horizon']))
        study.enqueue_trial({'alpha': float(np.clip(1.0, ALPHA_LOW, ALPHA_HIGH))})
        study.set_user_attr('run_id', run_id)
        study.set_user_attr('objective_name', 'WAPE + |Relative Bias|' if item['query_y'].sum() > 0 else 'MAE_zero_val_volume')
    return study

def completed_trials(study):
    return sum(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.get_trials(deepcopy=False))

def search_batch(items, batch):
    studies = [load_study(item) for item in items]
    counts = [completed_trials(study) for study in studies]
    while any(count < N_TRIALS for count in counts):
        trials = []
        alphas = []
        for study, count in zip(studies, counts):
            trial = study.ask() if count < N_TRIALS else None
            alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH, log=True) if trial is not None else study.best_params['alpha']
            trials.append(trial)
            alphas.append(alpha)
        coefficients = batch.coefficients(alphas, spectral=True)
        metrics, _ = batch.metrics(coefficients)
        for index, (item, study, trial) in enumerate(zip(items, studies, trials)):
            if trial is None:
                continue
            training = metrics['train'][index]
            validation = metrics['query'][index]
            trial.set_user_attr('train', training)
            trial.set_user_attr('val', validation)
            value = validation['score'] if validation['score'] is not None else validation['MAE']
            study.tell(trial, value)
            counts[index] += 1
            if counts[index] % CHECKPOINT_EVERY == 0 or counts[index] == N_TRIALS:
                atomic_joblib(RUN_ROOT / 'studies' / f"{item['key']}.joblib", study)
    alphas = [study.best_params['alpha'] for study in studies]
    spectral = batch.coefficients(alphas, spectral=True)
    direct = batch.coefficients(alphas, spectral=False)
    _, spectral_query = batch.metrics(spectral)
    metrics, query_predictions = batch.metrics(direct)
    host_coefficients = to_host(direct)
    display_rows = []
    for index, (item, study) in enumerate(zip(items, studies)):
        scale = max(1.0, float(np.abs(item['y']).max()))
        if not np.allclose(spectral_query[index], query_predictions[index], rtol=1e-6, atol=scale * 1e-7):
            raise ValueError('Спектральное и прямое решения Ridge расходятся')
        item['alpha'] = alphas[index]
        check_cpu_reference(item, host_coefficients[index], query_predictions[index])
        rows = []
        for trial in study.get_trials(deepcopy=False):
            if trial.state == optuna.trial.TrialState.COMPLETE:
                for period in ['train', 'val']:
                    rows.append(dict(trial=trial.number, alpha=trial.params['alpha'], period=period,
                                     route_id=item['route_id'], horizon=item['horizon'], **trial.user_attrs[period]))
        pd.DataFrame(rows).to_csv(RUN_ROOT / 'trials' / f"{item['key']}.csv", index=False)
        record = dict(run_id=run_id, route_index=item['route_index'], route_id=item['route_id'], horizon=item['horizon'],
                      alpha=item['alpha'], objective_name=study.user_attrs['objective_name'], objective_value=study.best_value,
                      best_trial=study.best_trial.number, completed_trials=counts[index],
                      train=metrics['train'][index], val=metrics['query'][index])
        atomic_json(RUN_ROOT / 'params' / f"{item['key']}.json", record)
        display_rows.extend(dict(route_id=item['route_id'], horizon=item['horizon'], period=period, **record[period]) for period in ['train', 'val'])
    metric_display.update(pd.DataFrame(display_rows))

def save_evaluation(items, batch, coefficients):
    metrics, predictions = batch.metrics(coefficients, include_original=True)
    host_coefficients = to_host(coefficients)
    display_rows = []
    for index, item in enumerate(items):
        check_cpu_reference(item, host_coefficients[index], predictions[index])
        rows = [dict(period=period, route_id=item['route_id'], horizon=item['horizon'], **metrics[source][index])
                for period, source in [('train', 'original_train'), ('train_val', 'train'), ('test', 'query')]]
        panel = item['query_frame'].copy()
        panel['horizon'] = item['horizon']
        panel['y_pred'] = predictions[index, :len(panel)]
        panel.to_parquet(RUN_ROOT / 'evaluation' / f"{item['key']}.parquet", index=False)
        atomic_json(RUN_ROOT / 'evaluation' / f"{item['key']}.json",
                    dict(run_id=run_id, route_id=item['route_id'], horizon=item['horizon'], metrics=rows, test_evaluated_before_full_refit=True))
        display_rows.extend(rows)
    metric_display.update(pd.DataFrame(display_rows))

def save_full(items, batch, coefficients):
    metrics, predictions = batch.metrics(coefficients)
    host_coefficients = to_host(coefficients)
    display_rows = []
    for index, item in enumerate(items):
        check_cpu_reference(item, host_coefficients[index], predictions[index])
        artifact = portable_model(item, host_coefficients[index])
        artifact.update(run_id=run_id, route_id=item['route_id'], route_index=item['route_index'], horizon=item['horizon'],
                        fit_through=item['fit_through'], history_through=str(end), versions=versions,
                        lag_table=item['lag_table'], prediction_clip_min=0.0, feature_pipeline_version=PIPELINE_VERSION,
                        fit_backend=BACKEND, matrix_dtype='float64')
        prediction = float(predictions[index, 0])
        model_path = MODEL_ROOT / f"h{item['horizon']}" / f"route_{item['route_index']:04d}.joblib"
        atomic_joblib(model_path, artifact)
        loaded = joblib.load(model_path)
        restored = np.maximum(loaded['model'].predict(item['query_x'] + item['x_mean']), 0)
        if not np.allclose(restored, predictions[index, :len(restored)], rtol=1e-10, atol=max(1., abs(prediction)) * 1e-10):
            raise ValueError('Прогноз сохранённой sklearn Ridge отличается от пакетного')
        full_row = dict(period='full_in_sample', route_id=item['route_id'], horizon=item['horizon'], **metrics['train'][index])
        active = np.asarray(artifact['preprocessing']['columns'])[artifact['preprocessing']['active_indices']]
        pd.DataFrame(dict(feature=active, coefficient_standardized=loaded['model'].coef_)).to_csv(RUN_ROOT / 'full' / f"{item['key']}_coefficients.csv", index=False)
        record = dict(run_id=run_id, route_id=item['route_id'], route_index=item['route_index'], horizon=item['horizon'],
                      timestamp=str(end + item['horizon'] * STEP), y_pred=prediction, metrics=full_row,
                      model_path=str(model_path.relative_to(OUTPUT_ROOT)), alpha=item['alpha'])
        atomic_json(RUN_ROOT / 'full' / f"{item['key']}.json", record)
        earlier = json.loads((RUN_ROOT / 'evaluation' / f"{item['key']}.json").read_text())['metrics']
        display_rows.extend([*earlier, full_row])
    metric_display.update(pd.DataFrame(display_rows))

def process_device_batch(items, phase):
    batch = DeviceBatch(items, spectral=phase == 'search')
    if phase == 'search':
        search_batch(items, batch)
    else:
        for item in items:
            settings = json.loads((RUN_ROOT / 'params' / f"{item['key']}.json").read_text())
            if settings['run_id'] != run_id:
                raise ValueError('Параметры относятся к другому запуску')
            item['alpha'] = settings['alpha']
        coefficients = batch.coefficients([item['alpha'] for item in items], spectral=False)
        if phase == 'evaluation':
            save_evaluation(items, batch, coefficients)
        else:
            save_full(items, batch, coefficients)
    synchronize()

def run_search():
    run_phase('search')
    selected = []
    for route_index in range(len(all_routes)):
        for h in range(1, 9):
            record = json.loads((RUN_ROOT / 'params' / f'{pair_key(route_index, h)}.json').read_text())
            if record['run_id'] != run_id:
                raise ValueError('Параметры другого запуска')
            selected.append(record)
    atomic_json(RUN_ROOT / 'selection_frozen.json', dict(run_id=run_id, params=selected, test_used_for_selection=False))
    print('Все параметры зафиксированы. Test не использовался при подборе.')

with threadpool_limits(limits=BLAS_THREADS):
    run_search()


## Независимая оценка до финального переобучения

Каждая выбранная модель обучается на всех допустимых стартах до начала test, включая период val. Отдельно сохраняются метрики исходного train, train + val и test. Общая конкурсная метрика рассчитывается из суммарных абсолютных и знаковых ошибок всех строк, а не средним по маршрутам. Дополнительно есть разбивка по горизонтам и дням test.

Эта оценка получена до обучения на test. Даты test ранее использовались в других экспериментах проекта, поэтому это не новый слепой holdout. Не выбирайте настройки этого запуска по test.

In [ ]:
def run_evaluation():
    frozen_path = RUN_ROOT / 'selection_frozen.json'
    if not frozen_path.exists() or json.loads(frozen_path.read_text())['run_id'] != run_id:
        raise ValueError('Сначала завершите подбор всех параметров')
    run_phase('evaluation')

def aggregate_metrics(rows, group_columns):
    frame = pd.DataFrame(rows)
    totals = frame.groupby(group_columns, dropna=False, as_index=False)[['n', 'actual_sum', 'prediction_sum', 'absolute_error', 'signed_error']].sum()
    denominator = totals.actual_sum.where(totals.actual_sum.gt(0))
    totals['WAPE'] = totals.absolute_error / denominator
    totals['abs_relative_bias'] = totals.signed_error.abs() / denominator
    totals['score'] = totals.WAPE + totals.abs_relative_bias
    totals['MAE'] = totals.absolute_error / totals.n
    return totals

def evaluation_report():
    rows = []
    predictions = []
    for route_index in range(len(all_routes)):
        for h in range(1, 9):
            key = pair_key(route_index, h)
            record = json.loads((RUN_ROOT / 'evaluation' / f'{key}.json').read_text())
            if record['run_id'] != run_id:
                raise ValueError('Оценка другого запуска')
            rows.extend(record['metrics'])
            predictions.append(pd.read_parquet(RUN_ROOT / 'evaluation' / f'{key}.parquet'))
    pd.DataFrame(rows).to_csv(RUN_ROOT / 'reports' / 'metrics_by_pair.csv', index=False)
    overall = aggregate_metrics(rows, ['period'])
    by_horizon = aggregate_metrics(rows, ['period', 'horizon'])
    overall.to_csv(RUN_ROOT / 'reports' / 'independent_metrics.csv', index=False)
    by_horizon.to_csv(RUN_ROOT / 'reports' / 'independent_metrics_by_horizon.csv', index=False)
    panel = pd.concat(predictions, ignore_index=True)
    if len(panel) != len(all_routes) * 8 * TEST_DAYS or panel.duplicated(['route_id', 'horizon', 'cutoff']).any():
        raise ValueError('Неполная независимая оценка')
    panel.to_parquet(RUN_ROOT / 'reports' / 'test_predictions.parquet', index=False)
    daily = [metric_row(part.y, part.y_pred, 'test', date=str(day)) for day, part in panel.groupby(panel.cutoff.dt.date)]
    pd.DataFrame(daily).to_csv(RUN_ROOT / 'reports' / 'test_metrics_by_day.csv', index=False)
    display(overall)
    display(by_horizon)
    note = 'Независимая оценка test получена до переобучения на всей размеченной истории. Даты test ранее использовались в проекте; это не новый слепой holdout.'
    (RUN_ROOT / 'reports' / 'evaluation_note.txt').write_text(note, encoding='utf-8')
    atomic_json(RUN_ROOT / 'evaluation_complete.json', dict(run_id=run_id, pairs=len(all_routes) * 8, test_evaluated_before_full_refit=True))
    print(note)

with threadpool_limits(limits=BLAS_THREADS):
    run_evaluation()
evaluation_report()


## Финальное обучение и сохранение 8000 моделей

Настройки Optuna неизменны. Пакетные решения Ridge на GPU, заполнение пропусков, стандартизация и корреляционные лаги переобучаются на всей доступной размеченной истории. Таблица показывает прежнюю независимую оценку и отдельную in-sample строку полного fit. Повторная оценка независимого test по финальной модели не производится.

Артефакт joblib содержит sklearn Ridge, список признаков, imputer, отбор постоянных колонок, scaler, выбранные корреляционные лаги и метаданные. Загрузить: `artifact = joblib.load(path)`; прогноз по подготовленному feature-frame: `predict_model(artifact, frame)`. Для исходных рядов сначала постройте признаки теми же функциями и сохранённым словарём лагов. Словарь не нужно обучать заново при применении сохранённой модели. Исходная история нужна для лаговых признаков.
Коэффициенты GPU переносятся в sklearn Ridge без повторного CPU fit. Каждый сохранённый файл загружается обратно, и его прогноз сравнивается с прогнозом пакетного решения. CPU-эталон дополнительно проверяется на первых двух парах этапа.


In [ ]:
def run_full():
    marker = RUN_ROOT / 'evaluation_complete.json'
    if not marker.exists() or json.loads(marker.read_text())['run_id'] != run_id:
        raise ValueError('Финальное обучение разрешено только после полной независимой оценки')
    run_phase('full')

with threadpool_limits(limits=BLAS_THREADS):
    run_full()


## Submission и архив

`submission.csv` содержит исходный порядок id и неотрицательный y_pred. Файл создаётся после всех 8000 финальных fit. Архив включает обе папки, параметры, checkpoints Optuna, отчёты, модели и submission. На Kaggle скачайте `ridge_by_route_horizon.zip` из `/kaggle/working`. Упаковка больших результатов может занять дополнительное время; CREATE_ARCHIVE можно отключить.

In [ ]:
def finish_run():
    records = []
    for route_index in range(len(all_routes)):
        for h in range(1, 9):
            key = pair_key(route_index, h)
            record = json.loads((RUN_ROOT / 'full' / f'{key}.json').read_text())
            if record['run_id'] != run_id or not (OUTPUT_ROOT / record['model_path']).is_file():
                raise ValueError('Отсутствует финальная модель или найден другой запуск')
            records.append(record)
    predictions = pd.DataFrame([{name: r[name] for name in ['route_id', 'horizon', 'timestamp', 'y_pred']} for r in records])
    predictions['timestamp'] = pd.to_datetime(predictions.timestamp)
    submission = future_requests.merge(predictions[['route_id', 'timestamp', 'y_pred']], on=['route_id', 'timestamp'], how='left', validate='one_to_one', sort=False)
    if len(submission) != len(all_routes) * 8 or not submission.id.equals(future_requests.id) or not np.isfinite(submission.y_pred).all() or (submission.y_pred < 0).any():
        raise ValueError('Неполный или некорректный submission')
    submission = submission[['id', 'y_pred']]
    submission.to_csv(RUN_ROOT / 'submission.csv', index=False)
    submission.to_csv(OUTPUT_ROOT / 'submission.csv', index=False)
    inventory = pd.DataFrame([{name: r[name] for name in ['route_index', 'route_id', 'horizon', 'model_path', 'alpha']} for r in records])
    inventory.to_csv(RUN_ROOT / 'reports' / 'model_inventory.csv', index=False)
    full_metrics = aggregate_metrics([r['metrics'] for r in records], ['period'])
    full_metrics.to_csv(RUN_ROOT / 'reports' / 'full_in_sample_metrics.csv', index=False)
    independent = pd.read_csv(RUN_ROOT / 'reports' / 'independent_metrics.csv')
    display(pd.concat([independent, full_metrics], ignore_index=True))
    display(submission.head())
    print(f'Сохранено {len(inventory)} финальных Ridge в {MODEL_ROOT}')
    print('Независимая оценка test получена до переобучения на всей размеченной истории.')
    if CREATE_ARCHIVE:
        archive = OUTPUT_ROOT / 'ridge_by_route_horizon.zip'
        temporary = archive.with_name(archive.name + '.tmp')
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as stream:
            for folder in [MODEL_ROOT, RUN_ROOT]:
                for path in sorted(folder.rglob('*')):
                    if path.is_file() and not path.name.endswith('.tmp'):
                        stream.write(path, path.relative_to(OUTPUT_ROOT))
        temporary.replace(archive)
        print('Архив:', archive)

finish_run()
